# MV01 DDP-B16 — Fresh epoch-1 GPU acceptance (independent of MV00)

**Bounded to epoch 1 only**, two Tesla T4 GPUs, YOLO11s + SCConv-Early + VGRA-V1, frozen Split B, 100-epoch recipe (`batch=16`, `imgsz=1024`). The worker exits after the first complete G4 epoch boundary, then exports two-rank checkpoints, manifest, best.pt, and receipts. The result is a **nonofficial, one-epoch acceptance**, not the 100-epoch continuous run. B-TEST stays sealed.

Attach exactly the same seven original project input datasets plus `awaishit/resema-ddp-b16-acceptance` Git bundle dataset (eight inputs). **Do not attach MV00 checkpoint files**; MV01 starts fresh from its own pinned pretrained source. Use T4 × 2 and **Save Version → Save & Run All**. MV00 may run concurrently only if Kaggle permits the second GPU commit/session and quota.

Output: `/kaggle/working/MV01_B16_ACCEPTANCE_OUTPUT/g4/ACCEPTANCE.json`; preserve the Kaggle saved-version output. The next step is off-session G4 recovery and verified MV01 native resume from epoch 2 before initiating the continuous-runner sessions.


In [ ]:
WORKER_CODE = "\"\"\"Bounded MV01 two-T4 FRESH GPU acceptance (one epoch, NOT official training).\n\nUses the independently verified private C1-derived source stage and fixed DDP-B16\nrecipe. The worker calls native trainer.train(), then exits immediately after\na successfully published rank-paired G4 epoch boundary. It does not use the\nhistorical epoch-1 G4 from a different source identity. No B-TEST access.\n\nThis is a development acceptance executable, not a 100-epoch session runner.\nIts receipt must be checked on a separate Kaggle session before promoting it.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport os\nimport shutil\nimport subprocess\nimport sys\nfrom datetime import timedelta\nfrom pathlib import Path\n\nRUN_SCHEMA = \"DDP_B16_FRESH_ONE_EPOCH_ACCEPTANCE_V1\"\nOUTPUT_SCHEMA = \"DDP_B16_EPOCH1_G4_CANDIDATE_V1\"\n\n\nclass AcceptanceFailure(RuntimeError):\n    pass\n\n\nclass EpochBoundaryStop(Exception):\n    pass\n\n\ndef need(ok, reason):\n    if not ok:\n        raise AcceptanceFailure(\"DDP_B16_ACCEPTANCE_\" + reason)\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\"rb\") as stream:\n        for block in iter(lambda: stream.read(4 * 1024 * 1024), b\"\"):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef safe_output_path(raw, *, label):\n    \"\"\"Only fresh, non-aliased /kaggle/working paths; never sealed B-TEST.\"\"\"\n    need(type(raw) is str and raw.startswith(\"/kaggle/working/\"),\n         label + \"_ROOT\")\n    p = Path(raw)\n    need(\"..\" not in p.parts and \"\\\\\" not in raw and\n         p.as_posix() == raw and\n         not any(part.casefold() in {\"test\", \"b-test\", \"b_test\"}\n                 for part in p.parts), label + \"_CANONICAL\")\n    need(not p.exists() and not p.is_symlink() and\n         p.parent.is_dir() and\n         not any(parent.is_symlink() for parent in p.parents),\n         label + \"_MUST_BE_NEW\")\n    return p\n\n\ndef run_configuration(plan, project, g4_root, artifacts):\n    \"\"\"Review-only shape check; executed before any writes or CUDA use.\"\"\"\n    from research.r9_training.ddp_b16.ddp_worker import validate_plan\n    validate_plan(plan)\n    need(plan[\"arm\"] == \"MV01\" and plan[\"mode\"] == \"FRESH_FROM_EARLY\",\n         \"MV01_FRESH_ONLY\")\n    need(plan[\"b_test_access\"] == \"NONE\", \"B_TEST_SEALED\")\n    need(not Path(plan[\"stage_root\"]).exists(), \"PRIVATE_STAGE_NOT_NEW\")\n    roots = [\n        safe_output_path(project, label=\"PROJECT\"),\n        safe_output_path(g4_root, label=\"G4\"),\n        safe_output_path(artifacts, label=\"ARTIFACTS\"),\n        Path(plan[\"stage_root\"]),\n    ]\n    for i, root in enumerate(roots):\n        for other in roots[i + 1:]:\n            need(root != other and not root.is_relative_to(other) and\n                 not other.is_relative_to(root), \"OVERLAPPING_OUTPUT_ROOTS\")\n    checkout = Path(plan[\"checkout_root\"]).resolve(strict=True)\n    for root in roots:\n        need(not root.is_relative_to(checkout) and\n             not checkout.is_relative_to(root), \"OUTPUT_OVERLAPS_CHECKOUT\")\n    return roots\n\n\ndef read_plan(path):\n    from research.r9_training.ddp_b16.ddp_worker import _unique_json\n    return _unique_json(Path(path))\n\n\ndef _g4_output(trainer, root, artifact_root, capsule, identity, run_id):\n    \"\"\"Rank 0 exports a complete, locally verified next-epoch G4 set.\"\"\"\n    next_epoch = 1\n    name = f\"{run_id}_epoch{next_epoch:03d}_two_rank_manifest.json\"\n    source_manifest = root / name\n    need(source_manifest.is_file(), \"NEW_G4_MANIFEST_MISSING\")\n    need(sha256(source_manifest) == capsule[\"manifest_sha256\"],\n         \"NEW_G4_MANIFEST_SHA\")\n    payload = json.loads(source_manifest.read_text(\"utf-8\"))\n    need(payload.get(\"complete\") is True and\n         payload.get(\"identity\") == identity and\n         payload.get(\"cursor\", {}).get(\"next_epoch\") == 1 and\n         payload[\"cursor\"][\"optimizer_updates\"] > 0,\n         \"NEW_G4_IDENTITY_OR_CURSOR\")\n\n    output = artifact_root / \"g4\"\n    need(not output.exists(), \"OUTPUT_ALREADY_EXISTS\")\n    output.mkdir()\n    files = {name: capsule[\"manifest_sha256\"],\n             \"best.pt\": capsule[\"selected_best_sha256\"]}\n    for rank in (0, 1):\n        record = payload[\"ranks\"][str(rank)]\n        files[record[\"filename\"]] = record[\"sha256\"]\n        files[record[\"receipt_filename\"]] = record[\"receipt_sha256\"]\n    for filename, digest in files.items():\n        need(Path(filename).name == filename, \"BAD_G4_MEMBER_NAME\")\n        origin = Path(trainer.best) if filename == \"best.pt\" else root / filename\n        need(origin.is_file() and not origin.is_symlink() and\n             sha256(origin) == digest, \"G4_MEMBER_SHA_MISMATCH\")\n        target = output / filename\n        shutil.copyfile(origin, target)\n        need(sha256(target) == digest, \"G4_COPY_DRIFT\")\n    result = {\n        \"schema\": OUTPUT_SCHEMA,\n        \"status\": \"NONOFFICIAL_EPOCH1_G4_SAVED_NOT_OFFSESSION_VERIFIED\",\n        \"source_release_sha\": identity[\"ddp_release_sha\"],\n        \"run_id\": run_id,\n        \"next_epoch\": 1,\n        \"optimizer_updates\": payload[\"cursor\"][\"optimizer_updates\"],\n        \"manifest_sha256\": capsule[\"manifest_sha256\"],\n        \"selected_best_sha256\": capsule[\"selected_best_sha256\"],\n        \"files_sha256\": files,\n        \"official_training_authorized\": False,\n        \"offsession_resume_verified\": False,\n        \"b_test_access\": \"NONE\",\n    }\n    receipt = output / \"ACCEPTANCE.json\"\n    receipt.write_text(json.dumps(result, indent=2, sort_keys=True) + \"\\n\",\n                       encoding=\"utf-8\")\n    return result\n\n\ndef child(plan, *, project, g4_root, artifacts):\n    import torch\n    import torch.distributed as dist\n\n    from research.r9_training.ddp_b16.ddp_worker import review_worker_inputs\n    from research.r9_training.ddp_b16.ddp_verified_admission import verify_staged_admission\n    from research.r9_training.ddp_b16.ddp_rank_admission import require_review_consensus\n    from research.r9_training.ddp_b16.ddp_native_binding import make_native_binding\n    from research.r9_training.ddp_b16.ddp_native_binding_consensus import require_native_binding_consensus\n    from research.r9_training.ddp_b16.ddp_native_bootstrap_candidate import (\n        verify_two_t4_runtime, load_pinned_native_class, bind_candidate_fresh_state,\n    )\n    from research.r9_training.ddp_b16 import ddp_native_g4_hook as hook\n\n    rank = int(os.environ[\"RANK\"])\n    local = int(os.environ[\"LOCAL_RANK\"])\n    world = int(os.environ[\"WORLD_SIZE\"])\n    need(os.environ.get(\"CUDA_VISIBLE_DEVICES\") == \"0,1\" and\n         (world, rank, local) in {(2, 0, 0), (2, 1, 1)}, \"T4_RANK_MAPPING\")\n    need(Path(plan[\"stage_root\"]).is_dir() and\n         Path(g4_root).is_dir(), \"PREPARED_STAGE_OR_G4_MISSING\")\n    need(not Path(project).exists(), \"NATIVE_PROJECT_ALREADY_EXISTS\")\n\n    dist.init_process_group(\"gloo\", init_method=\"env://\",\n                            rank=rank, world_size=2,\n                            timeout=timedelta(seconds=600))\n    try:\n        review = review_worker_inputs(plan, stage_source=False)\n        admission = verify_staged_admission(plan, review)\n        require_review_consensus(admission)\n        config = read_plan(plan[\"r0_input_config\"])\n        binding = make_native_binding(plan, admission, config,\n                                      project=project)\n        require_native_binding_consensus(binding)\n        need(binding[\"identity\"][\"ddp_release_sha\"] ==\n             plan[\"expected_git_head\"], \"RELEASE_IDENTITY\")\n        need(binding[\"arm\"] == \"MV01\" and\n             binding[\"overrides\"][\"epochs\"] == 100 and\n             binding[\"overrides\"][\"batch\"] == 16 and\n             binding[\"overrides\"][\"resume\"] is False, \"FROZEN_RECIPE\")\n        verify_two_t4_runtime(torch.cuda)\n        native = load_pinned_native_class(binding)\n        need(native.__name__ == \"VGRADetectionTrainer\", \"NATIVE_CLASS\")\n        trainer = native(**binding[\"constructor\"],\n                         overrides=binding[\"overrides\"])\n        need(trainer.world_size == 2 and trainer.ddp is False,\n             \"NATIVE_WORLD_MAPPING\")\n        dist.barrier()\n    finally:\n        if dist.is_initialized():\n            dist.destroy_process_group()\n\n    # This separate candidate binding is not inferred from historical batch-8 G4.\n    bind_candidate_fresh_state(trainer, binding, Path(g4_root))\n    original = hook.ddp_b16_g4_epoch_end\n    capsule = {}\n\n    def one_boundary(obj, *, epoch, nb, last_opt_step):\n        need(epoch == 0, \"EXPECTED_FRESH_EPOCH1\")\n        result = original(obj, epoch=epoch, nb=nb,\n                          last_opt_step=last_opt_step)\n        need(result.get(\"status\") ==\n             \"LOCAL_SYNCHRONIZED_DDP_CAPSULE_EXTERNAL_AUTHENTICATION_PENDING\",\n             \"NO_COMPLETE_NONTERMINAL_G4\")\n        capsule.update(result)\n        dist.barrier()\n        raise EpochBoundaryStop(\"EPOCH1_G4_BOUNDARY_COMPLETE\")\n\n    hook.ddp_b16_g4_epoch_end = one_boundary\n    try:\n        try:\n            trainer.train()\n            raise AcceptanceFailure(\"UNEXPECTED_FULL_TRAINING_RETURN\")\n        except EpochBoundaryStop:\n            need(bool(capsule.get(\"manifest_sha256\")), \"G4_NOT_PUBLISHED\")\n            record = {\n                \"rank\": rank,\n                \"manifest_sha256\": capsule[\"manifest_sha256\"],\n                \"best_sha256\": capsule[\"selected_best_sha256\"],\n            }\n            peers = [None, None]\n            dist.all_gather_object(peers, record)\n            need([p[\"rank\"] for p in peers] == [0, 1] and\n                 peers[0][\"manifest_sha256\"] == peers[1][\"manifest_sha256\"] and\n                 peers[0][\"best_sha256\"] == peers[1][\"best_sha256\"],\n                 \"G4_RANK_CONSENSUS\")\n            if rank == 0:\n                result = _g4_output(trainer, Path(g4_root),\n                                    Path(artifacts), capsule,\n                                    binding[\"identity\"], plan[\"run_id\"])\n                print(\"DDP_B16_EPOCH1_G4_RESULT=\" +\n                      json.dumps(result, sort_keys=True), flush=True)\n            dist.barrier()\n            receipt = Path(artifacts) / \"g4\" / \"ACCEPTANCE.json\"\n            need(receipt.is_file(), \"ACCEPTANCE_RECEIPT_NOT_SHARED\")\n            dist.barrier()\n    finally:\n        hook.ddp_b16_g4_epoch_end = original\n        if dist.is_initialized():\n            dist.destroy_process_group()\n\n\ndef parent(plan_path, *, project, g4_root, artifacts):\n    from research.r9_training.ddp_b16.ddp_worker import review_worker_inputs\n    from research.r9_training.ddp_b16.ddp_verified_admission import verify_staged_admission\n\n    plan = read_plan(plan_path)\n    run_configuration(plan, project, g4_root, artifacts)\n    need(os.environ.get(\"CUDA_VISIBLE_DEVICES\") == \"0,1\",\n         \"CUDA_0_1_REQUIRED\")\n    need(not any(name in os.environ for name in\n                 (\"RANK\", \"LOCAL_RANK\", \"WORLD_SIZE\")),\n         \"PARENT_INSIDE_TORCHRUN\")\n\n    # One-time source/asset admission before launching two GPU ranks.\n    review = review_worker_inputs(plan, stage_source=True)\n    admission = verify_staged_admission(plan, review)\n    need(admission[\"identity\"][\"ddp_release_sha\"] ==\n         plan[\"expected_git_head\"], \"STAGE_RELEASE_DRIFT\")\n    Path(g4_root).mkdir()\n    Path(artifacts).mkdir()\n    command = [\n        sys.executable, \"-m\", \"torch.distributed.run\",\n        \"--standalone\", \"--nnodes=1\", \"--nproc_per_node=2\",\n        \"--max_restarts=0\", str(Path(__file__).resolve()),\n        \"--plan\", str(plan_path), \"--project\", project,\n        \"--g4-root\", g4_root, \"--artifacts\", artifacts, \"--child\",\n    ]\n    env = dict(os.environ)\n    env[\"OMP_NUM_THREADS\"] = \"1\"\n    env[\"PYTHONDONTWRITEBYTECODE\"] = \"1\"\n    checkout = str(Path(plan[\"checkout_root\"]).resolve(strict=True))\n    env[\"PYTHONPATH\"] = checkout + os.pathsep + env.get(\"PYTHONPATH\", \"\")\n    log_path = Path(artifacts) / \"TWO_RANK_EPOCH1.log\"\n    with log_path.open(\"x\", encoding=\"utf-8\") as log:\n        process = subprocess.run(command, cwd=checkout, env=env,\n                                 stdout=log, stderr=subprocess.STDOUT,\n                                 check=False)\n    need(process.returncode == 0,\n         \"TWO_RANK_FAILED_READ_\" + str(log_path))\n    receipt_path = Path(artifacts) / \"g4\" / \"ACCEPTANCE.json\"\n    need(receipt_path.is_file(), \"MISSING_EPOCH1_ACCEPTANCE_RECEIPT\")\n    receipt = json.loads(receipt_path.read_text(\"utf-8\"))\n    need(receipt.get(\"status\") ==\n         \"NONOFFICIAL_EPOCH1_G4_SAVED_NOT_OFFSESSION_VERIFIED\" and\n         receipt.get(\"source_release_sha\") == plan[\"expected_git_head\"] and\n         receipt.get(\"b_test_access\") == \"NONE\", \"ACCEPTANCE_RECEIPT_INVALID\")\n    for name, digest in receipt[\"files_sha256\"].items():\n        candidate = Path(artifacts) / \"g4\" / name\n        need(candidate.is_file() and sha256(candidate) == digest,\n             \"OUTPUT_CHECKSUM_FAILED\")\n    print(\"DDP_B16_GPU_ACCEPTANCE_CANDIDATE_EPOCH1_SAVED\")\n    print(\"OUTPUT_ROOT=\" + str(Path(artifacts) / \"g4\"))\n    print(\"MANIFEST_SHA256=\" + receipt[\"manifest_sha256\"])\n    print(\"B_TEST=SEALED; OFFSESSION_RESUME=UNVERIFIED\")\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--plan\", required=True)\n    parser.add_argument(\"--project\", required=True)\n    parser.add_argument(\"--g4-root\", required=True)\n    parser.add_argument(\"--artifacts\", required=True)\n    mode = parser.add_mutually_exclusive_group(required=True)\n    mode.add_argument(\"--accept-one-epoch-gpu\", action=\"store_true\")\n    mode.add_argument(\"--child\", action=\"store_true\", help=argparse.SUPPRESS)\n    args = parser.parse_args(argv)\n    plan = read_plan(args.plan)\n    if args.child:\n        child(plan, project=args.project, g4_root=args.g4_root,\n              artifacts=args.artifacts)\n    else:\n        parent(args.plan, project=args.project, g4_root=args.g4_root,\n               artifacts=args.artifacts)\n\n\nif __name__ == \"__main__\":\n    main()\n"

# MV01 bounded epoch-1 acceptance: one code cell after editing the 3 pins below.
# This is NOT the final 100-epoch production notebook.
BUNDLE_DATASET_ROOT = "/kaggle/input/datasets/awaishit/resema-ddp-b16-acceptance"
EXPECTED_GIT_HEAD = "f440d329d4544ec08ad63c1972bcf9a8b8a7eed8"
EXPECTED_BUNDLE_SHA256 = "4dc26b7277d58ed0b1fbcc1f4fac68aaf7ad417951e735bc8be2b040764861bb"

import hashlib
import importlib.util
import json
import os
import re
import subprocess
import sys
from pathlib import Path

import torch

assert re.fullmatch(r"[0-9a-f]{40}", EXPECTED_GIT_HEAD), "SET_EXACT_RELEASE_GIT_SHA"
assert re.fullmatch(r"[0-9a-f]{64}", EXPECTED_BUNDLE_SHA256), "SET_EXACT_BUNDLE_SHA"
assert torch.cuda.is_available() and torch.cuda.device_count() == 2, "REQUIRE_T4_X2"
assert all("T4" in torch.cuda.get_device_name(i).upper() for i in (0, 1)), "NOT_T4_X2"

inputs = {
    "n1": Path("/kaggle/input/datasets/awaishit/detection-frozen-git-n1-accepted-r1"),
    "handoff": Path("/kaggle/input/datasets/awaishit/detection-project-training-handoff-r1"),
    "legacy": Path("/kaggle/input/datasets/awaishit/detection-legacy-notebook-files-flat-r1"),
    "split_b": Path("/kaggle/input/datasets/utopianstar/grazpedwri-dx-split-b"),
    "assets": Path("/kaggle/input/datasets/nithoukhan/d4e-r12-accepted-assets"),
    "archives": Path("/kaggle/input/datasets/nithoukhan/d4e-r12-raw-archives-v1"),
    "evidence": Path("/kaggle/input/datasets/nithoukhan2/d4f-r7d-legacy-exact-source-evidence-r1"),
}
assert all(root.is_dir() for root in inputs.values()), "MOUNT_ALL_EXISTING_7_DATASETS"
bundle = Path(BUNDLE_DATASET_ROOT) / "RESEMA_DDP_B16_ACCEPTANCE.bundle"
assert bundle.is_file() and not bundle.is_symlink(), "MOUNT_NEW_SOURCE_BUNDLE_DATASET"

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(4 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

assert sha256(bundle) == EXPECTED_BUNDLE_SHA256, "OFFLINE_SOURCE_BUNDLE_PIN_MISMATCH"

work = Path("/kaggle/working/MV01_B16_ACCEPTANCE_PREP")
repo = work / "repo"
stage = Path("/kaggle/working/MV01_B16_FRESH_STAGE")
g4_root = Path("/kaggle/working/MV01_B16_FRESH_G4")
project = Path("/kaggle/working/MV01_B16_NATIVE_PROJECT")
artifacts = Path("/kaggle/working/MV01_B16_ACCEPTANCE_OUTPUT")
run_id = "MV01_B16_FRESH_ACCEPTANCE"
assert not any(p.exists() for p in (work, stage, g4_root, project, artifacts)), "FRESH_OUTPUTS_REQUIRED"

work.mkdir()
worker_path = work / "MV01_FRESH_EPOCH1_WORKER.py"
worker_path.write_text(WORKER_CODE, encoding="utf-8")
print("MV01_ACCEPTANCE_WORKER_SHA256:", sha256(worker_path),flush=True)
subprocess.run(["git", "clone", "--quiet", str(bundle), str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--quiet", "--detach", EXPECTED_GIT_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_GIT_HEAD, "SOURCE_COMMIT_MISMATCH"
assert not subprocess.check_output(["git", "-C", str(repo), "status", "--porcelain", "--untracked-files=no"], text=True).strip()

sys.dont_write_bytecode = True
sys.path.insert(0, str(repo))
from research.r9_training.ddp_b16.r0_relocation import restore_r0_config
from research.r9_training.ddp_b16.ddp_worker import _unique_json

configs = sorted(inputs["n1"].rglob("R0_RELOCATED_CONFIG_CANDIDATE.json"))
assert len(configs) == 1, "R0_CONFIG_MISSING_OR_AMBIGUOUS"
config = configs[0]
r0_file = (repo / "research/r9_training/frozen_source/ultralytics/models"
           "/yolo/detect/d4e_r3_read_only_identity_preflight.py")
spec = importlib.util.spec_from_file_location("_mv01_original_r0", r0_file)
r0 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(r0)

source_roots = [inputs[k] for k in ("n1", "handoff", "legacy", "assets", "archives", "evidence")]
restored = restore_r0_config(
    config, source_roots,
    asset_pins=r0.ASSET_PINS, overlay_pins=r0.OVERLAY_PINS,
    code_pins=r0.R21_PINS, native_blob_pin=r0.NATIVE_BLOB_SHA1,
    runtime_yaml_pin="f2cc6e31f5f1be152f4dd7603a397e142963923f1d35de613eb7c61b168bf0c0",
    mirrored_handoff=inputs["handoff"],
    archives_roots=[inputs["assets"], inputs["archives"]],
)
r0_receipt = r0.verify_inputs(
    _unique_json(config), "MV01", run_id,
    "f2cc6e31f5f1be152f4dd7603a397e142963923f1d35de613eb7c61b168bf0c0",
    str(repo / "research/r9_training/R9_RELEASE_RECIPE_CANDIDATE_R1.json"),
)
assert r0_receipt["status"] == "PASS_LOCAL_FILES_MATCH_PINNED_BYTES_NOT_GPU_ACCEPTANCE"
assert r0_receipt["b_test_access"] == "NONE"
print("R0 ASSETS VERIFIED; FILES RELOCATED:", restored["restored"], flush=True)

plan = {
    "schema": "VGRA_DDP_B16_WORKER_PLAN_REVIEW_V1",
    "arm": "MV01",
    "run_id": run_id,
    "mode": "FRESH_FROM_EARLY",
    "checkout_root": str(repo),
    "expected_git_head": head,
    "r0_input_config": str(config),
    "stage_root": str(stage),
    "b_test_access": "NONE",
}
plan_path = work / "plan.json"
plan_path.write_text(json.dumps(plan, indent=2, sort_keys=True) + "\n", encoding="utf-8")
env = dict(os.environ)
env.update({
    "CUDA_VISIBLE_DEVICES": "0,1",
    "OMP_NUM_THREADS": "1",
    "PYTHONDONTWRITEBYTECODE": "1",
    "PYTHONPATH": str(repo) + os.pathsep + env.get("PYTHONPATH", ""),
})
command = [
    sys.executable, "-m",
    str(worker_path),
    "--plan", str(plan_path), "--project", str(project),
    "--g4-root", str(g4_root), "--artifacts", str(artifacts),
    "--accept-one-epoch-gpu",
]
print("RUNNING: MV01 FRESH NONOFFICIAL EPOCH-1 ACCEPTANCE", flush=True)
print("GIT:", head, "| T4x2 | global batch 16 | 1024 | B-TEST SEALED", flush=True)
result = subprocess.run(command, cwd=repo, env=env, check=False)
assert result.returncode == 0, "ACCEPTANCE_FAILED_INSPECT_" + str(artifacts / "TWO_RANK_EPOCH1.log")
receipt = json.loads((artifacts / "g4" / "ACCEPTANCE.json").read_text("utf-8"))
assert receipt["status"] == "NONOFFICIAL_EPOCH1_G4_SAVED_NOT_OFFSESSION_VERIFIED"
print("MV01 EPOCH1 G4 CANDIDATE COMPLETE")
print("SAVED OUTPUT:", artifacts)
print("EXTERNAL MANIFEST SHA256:", receipt["manifest_sha256"])
print("SAVE VERSION OUTPUT AND VERIFY IN A SEPARATE SESSION BEFORE OFFICIAL TRAINING")
